# Star Type Classification Dataset

**Gruppenprojekt Data Science, 2. Semester**

Ziel dieses Notebooks ist es, das Star Type Classification Dataset zu laden, zu explorieren, Datenprobleme zu dokumentieren, eine reproduzierbare Preprocessing-Pipeline zu entwickeln und Machine-Learning-Modelle fuer die Vorhersage der Sternklasse `Type` zu trainieren und zu evaluieren.

## 1. Problemstellung

Das Dataset enthaelt physikalische und spektrale Eigenschaften von Sternen. Die zentrale Machine-Learning-Aufgabe ist eine **multiklassige Klassifikation**: Aus den Merkmalen `Temperature`, `L`, `R`, `A_M`, `Color` und `Spectral_Class` soll die Zielvariable `Type` vorhergesagt werden.

Die Klassen sind:

| Type | Bedeutung |
| --- | --- |
| 0 | Red Dwarf |
| 1 | Brown Dwarf |
| 2 | White Dwarf |
| 3 | Main Sequence |
| 4 | Super Giants |
| 5 | Hyper Giants |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 2. Dataset laden

Die Datei ist nicht sauber als Standard-CSV gespeichert:

- Trennzeichen ist `~`, nicht `,`.
- Die erste Zeile ist eine Beschreibung und nicht der Header.
- Rechts neben den relevanten Spalten befinden sich leere Zusatzspalten.

Deshalb wird die Datei gezielt geladen: `sep='~'`, `skiprows=1`, danach werden nur die ersten sieben Spalten behalten.

In [ ]:
DATA_PATH = Path("nasa_stars_data.csv")

raw = pd.read_csv(DATA_PATH, sep="~", skiprows=1, header=0, usecols=range(7), engine="python")
raw.columns = ["Temperature", "L", "R", "A_M", "Color", "Spectral_Class", "Type"]

raw.head()

In [ ]:
raw.shape

## 3. Erste Datenpruefung

Wir pruefen Datentypen, fehlende Werte, Zielklassenverteilung und auffaellige Kategorien.

In [ ]:
raw.info()

In [ ]:
raw.isna().sum()

In [ ]:
raw["Type"].value_counts().sort_index()

In [ ]:
raw["Color"].value_counts(dropna=False)

In [ ]:
raw["Spectral_Class"].value_counts(dropna=False)

In [ ]:
numeric_cols = ["Temperature", "L", "R", "A_M"]
raw[numeric_cols].describe().T

### Befunde aus der Exploration

- Das Dataset enthaelt 240 Beobachtungen und ist bei der Zielvariable perfekt balanciert: jede Klasse kommt 40-mal vor.
- Es gibt fehlende Werte in `Temperature`, `R`, `Color` und `Spectral_Class`.
- `Color` ist uneinheitlich kodiert, z. B. `Blue-white`, `Blue White`, `Blue white`, `Blue-White`, `Blue-wh!te`, `Blu`, `white` und ` White`.
- Die numerischen Variablen `L` und `R` besitzen sehr unterschiedliche Groessenordnungen und starke Spannweiten. Skalierung ist daher besonders fuer lineare Modelle und SVM wichtig.
- Die Originaldatei enthaelt technische Ladeprobleme: falsches Standard-Trennzeichen, vorgeschaltete Textzeile und leere Zusatzspalten.

## 4. Visualisierung

Die folgenden Diagramme helfen, Unterschiede zwischen den Sternklassen sichtbar zu machen.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=raw, x="Type", y=col, ax=ax)
    ax.set_title(f"{col} nach Sternklasse")

plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=raw, x="Temperature", y="A_M", hue="Type", palette="tab10")
plt.title("Temperatur und absolute Magnitude nach Sternklasse")
plt.tight_layout()

## 5. Cleaning-Strategie

Die Originaldatei wird nicht manuell ueberschrieben. Stattdessen wird eine reproduzierbare Cleaning-Funktion definiert:

- Leere Strings werden als fehlende Werte behandelt.
- Numerische Spalten werden sicher in Zahlen konvertiert.
- `Color` wird vereinheitlicht, z. B. Schreibfehler und verschiedene Schreibweisen werden zusammengefuehrt.
- `Spectral_Class` wird bereinigt und leere Werte werden spaeter im Pipeline-Schritt imputiert.
- Fehlende numerische Werte werden mit dem Median imputiert.
- Kategoriale Werte werden mit dem haeufigsten Wert imputiert und per One-Hot-Encoding kodiert.

In [ ]:
def clean_star_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    for col in cleaned.columns:
        if cleaned[col].dtype == "object":
            cleaned[col] = cleaned[col].astype("string").str.strip()
            cleaned[col] = cleaned[col].replace({"": pd.NA})

    for col in ["Temperature", "L", "R", "A_M", "Type"]:
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    color_map = {
        "blu": "Blue",
        "blue": "Blue",
        "blue white": "Blue-white",
        "blue-white": "Blue-white",
        "blue-white": "Blue-white",
        "blue-wh!te": "Blue-white",
        "white": "White",
        "whitish": "White",
        "white-yellow": "Yellow-white",
        "yellowish white": "Yellow-white",
        "yellow-white": "Yellow-white",
        "yellowish": "Yellowish",
        "pale yellow orange": "Pale yellow orange",
        "orange": "Orange",
        "orange-red": "Orange-red",
        "red": "Red",
    }

    normalized_color = cleaned["Color"].str.lower().str.replace("_", " ", regex=False).str.replace(r"\s+", " ", regex=True)
    cleaned["Color"] = normalized_color.map(color_map).fillna(cleaned["Color"])

    cleaned["Spectral_Class"] = cleaned["Spectral_Class"].str.upper()
    cleaned[["Color", "Spectral_Class"]] = cleaned[["Color", "Spectral_Class"]].astype(object).where(
        cleaned[["Color", "Spectral_Class"]].notna(), np.nan
    )
    cleaned["Type"] = cleaned["Type"].astype("Int64")

    return cleaned

data = clean_star_dataframe(raw)
data.head()

In [ ]:
data.isna().sum()

In [ ]:
data["Color"].value_counts(dropna=False)

## 6. Machine-Learning-Pipeline

Wir verwenden eine `ColumnTransformer`-Pipeline. Das verhindert Data Leakage, weil Imputation, Skalierung und One-Hot-Encoding nur auf den Trainingsdaten gelernt werden und anschliessend auf Testdaten angewendet werden.

In [ ]:
X = data.drop(columns="Type")
y = data["Type"].astype(int)

numeric_features = ["Temperature", "L", "R", "A_M"]
categorical_features = ["Color", "Spectral_Class"]

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocess, numeric_features),
    ("cat", categorical_preprocess, categorical_features),
])

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1_macro"], return_train_score=False)
    results.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
    })

results_df = pd.DataFrame(results).sort_values("f1_macro_mean", ascending=False)
results_df

## 7. Finales Modell evaluieren

Nach dem Modellvergleich trainieren wir das beste Modell auf einem Trainingssplit und bewerten es auf einem separaten Testsplit. Wegen der kleinen Datenmenge ist Cross Validation wichtiger als ein einzelner Split; der Split hilft aber, eine konkrete Confusion Matrix und einen Classification Report zu zeigen.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

final_pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", best_model)])
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

print(f"Bestes Modell laut Cross Validation: {best_model_name}")
print(f"Accuracy Testset: {accuracy_score(y_test, y_pred):.3f}")
print(f"Macro F1 Testset: {f1_score(y_test, y_pred, average='macro'):.3f}")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.xlabel("Vorhergesagte Klasse")
plt.ylabel("Tatsaechliche Klasse")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()

## 8. Interpretation und Fazit

Die Sternklassifikation eignet sich gut als ueberwachtes Lernproblem, weil die Zielvariable `Type` vollstaendig vorhanden und balanciert ist. Die wichtigsten Vorverarbeitungsschritte sind das robuste Laden der Datei, die Vereinheitlichung von Farbkategorien, die Behandlung fehlender Werte und die Skalierung numerischer Features.

Fuer die Bewertung werden **Accuracy** und **Macro F1** verwendet. Accuracy ist wegen der balancierten Klassen gut interpretierbar. Macro F1 ist trotzdem sinnvoll, weil jede Klasse gleich stark gewichtet wird und damit schwache Leistung in einzelnen Klassen sichtbar bleibt.

Aufgrund der kleinen Datenmenge von 240 Beobachtungen sollte die Modellqualitaet nicht nur ueber einen einzelnen Train-Test-Split beurteilt werden. Die 5-fache stratified Cross Validation liefert eine stabilere Schaetzung.

## 9. Grenzen und moegliche Verbesserungen

- Das Dataset ist klein, wodurch Ergebnisse je nach Split schwanken koennen.
- Einige fehlende Werte koennten astrophysikalisch fundierter imputiert werden, z. B. anhand aehnlicher Sternklassen oder physikalischer Beziehungen.
- `Color` enthaelt Schreibfehler und uneinheitliche Labels. Die hier genutzte Mapping-Tabelle ist transparent, aber fachlich diskutierbar.
- Eine Hyperparameter-Optimierung mit `GridSearchCV` oder `RandomizedSearchCV` koennte die Modelle weiter verbessern.
- Feature Engineering mit logarithmierten Werten fuer `L` und `R` koennte sinnvoll sein, weil diese Variablen sehr grosse Wertebereiche besitzen.

## 10. Dokumentation der LLM-Nutzung

Fuer dieses Projekt wurde ein Large Language Model als Unterstuetzung verwendet. Genutzt wurde es fuer:

- Strukturierung des Notebooks und des Abschlussberichts
- Vorschlaege fuer eine geeignete Machine-Learning-Strategie
- Formulierung einer reproduzierbaren Cleaning- und Preprocessing-Pipeline
- Hinweise zur Dokumentation typischer Datenprobleme

Die Verantwortung fuer Code, Interpretation und finale Abgabe liegt vollstaendig bei der Gruppe. Alle Gruppenmitglieder muessen die Schritte im Notebook nachvollziehen und erklaeren koennen.